# Frontend Technical Requirements

This notebook defines the frontend implementation requirements for the Deploy Tool UI.

## 1. Stack and Project Structure

- Build tool: **Vite**
- Framework: **React + TypeScript**
- Styling: **Tailwind CSS**
- Frontend root: `/frontend`

### Required structure

```
frontend/
├── package.json
├── vite.config.ts
├── tailwind.config.js
├── postcss.config.js
├── tsconfig.json
├── index.html
├── src/
│   ├── main.tsx
│   ├── App.tsx
│   ├── index.css
│   ├── components/
│   │   ├── RepoSelector.tsx
│   │   ├── BranchDropdown.tsx
│   │   ├── FilerIPInput.tsx
│   │   ├── DeployButton.tsx
│   │   ├── LogViewer.tsx
│   │   └── StatusIndicator.tsx
│   ├── hooks/
│   │   ├── useWebSocket.ts
│   │   ├── useDeploy.ts
│   │   ├── useBranches.ts
│   │   └── useQueryParams.ts
│   ├── api/
│   │   └── client.ts
│   └── types/
│       └── index.ts
└── frontend_requirements.ipynb
```

## 2. Component Hierarchy

```
App
├── Header
│   └── StatusIndicator
├── RepoSelector
├── BranchDropdown
├── FilerIPInput
├── DeployButton
└── LogViewer
```

App owns shared deployment form state (`repo`, `branch`, `filerIP`) and deployment lifecycle state.

## 3. `RepoSelector` Requirements

- Supports exactly two repositories: `nbn-daemon`, `unity`
- Interaction:
  - selecting repo updates URL query param
  - clears selected branch
  - triggers branch refetch
- Tailwind segmented-control style

## 4. `BranchDropdown` (Custom Searchable Dropdown)

Must be custom-built (no UI library).

### Behavior
- Click opens dropdown
- Search input filters list by case-insensitive substring
- Arrow up/down changes highlighted option
- Enter selects highlighted option
- Escape closes dropdown
- Click outside closes dropdown
- Loading state while branches are fetched
- Empty state: `No branches match`

### Props

```ts
interface BranchDropdownProps {
  branches: string[]
  selected: string | null
  onSelect: (branch: string) => void
  loading: boolean
}
```

## 5. `FilerIPInput` Requirements

- Standard text input
- On blur, validate with regex: `^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$`
- Show inline error when invalid
- Changes update query param immediately

## 6. `DeployButton` Requirements

- States:
  - idle: enabled
  - loading: spinner + `Deploying...`
  - disabled: missing required values or active deploy for same repo
- On click:
  1. Call `POST /api/deploy`
  2. Use returned `deploymentId` to start log websocket

## 7. `LogViewer` Requirements

Terminal-style panel with Tailwind classes such as:
`bg-gray-900 text-gray-100 font-mono text-sm p-4 overflow-y-auto`

### Rendering rules
- stdout: gray/white
- stderr: red/orange
- system: cyan + italic

### Behavior
- Auto-scroll to bottom on new lines
- Pause auto-scroll when user scrolls up
- Resume when user reaches bottom
- Show `Connecting...` before websocket opens
- Show final success/failure message when deploy completes

## 8. `StatusIndicator` Requirements

Displays one of: `Idle | Running | Success | Failed`

- Idle: gray badge
- Running: blue badge + spinner
- Success: green badge
- Failed: red badge

## 9. State Management

Use React hooks only (no global state library).

- `useState` for local view/form state
- custom hooks for domain state:
  - `useQueryParams` for URL sync
  - `useBranches` for branch loading
  - `useDeploy` for deployment start/status
  - `useWebSocket` for logs

## 10. API Integration Layer

`src/api/client.ts` centralizes REST calls with `fetch` wrapper and error handling.

```ts
const API_BASE = '/api'

async function deploy(repo: string, branch: string, filerIP: string): Promise<{ deploymentId: string }>
async function getBranches(repo: string): Promise<string[]>
async function getStatus(): Promise<{ status: string; repos: Record<string, boolean>; docker: boolean }>
async function getDeployment(id: string): Promise<DeploymentStatus>
```

## 11. URL Query Parameter Synchronization

`useQueryParams` contract:

```ts
interface DeployParams {
  repo: 'nbn-daemon' | 'unity' | null
  branch: string | null
  filerIP: string | null
}

function useQueryParams(): [DeployParams, (params: Partial<DeployParams>) => void]
```

### Required behavior
- Read params on initial load
- Pre-fill UI fields from URL
- Update URL using `history.replaceState()`
- Keep params optional
- URL format example:
  `/?repo=nbn-daemon&branch=feature/x&filerIP=10.0.0.100`

## 12. WebSocket Hook Requirements

`useWebSocket(deploymentId)` connects when deployment ID exists.

- endpoint: `ws://localhost:8000/api/ws/logs/{deploymentId}`
- stores incoming log messages
- marks `done=true` when message has `done: true`
- closes and reconnects when deployment ID changes
- closes on unmount

## 13. TypeScript Types

`src/types/index.ts` includes:

```ts
type RepoName = 'nbn-daemon' | 'unity'
interface DeployRequest { repo: RepoName; branch: string; filerIP: string }
interface DeployResponse { deploymentId: string; status: 'started' }
interface DeploymentStatus {
  id: string
  repo: RepoName
  branch: string
  filerIP: string
  status: 'running' | 'success' | 'failed'
  exitCode: number | null
  startedAt: string
  completedAt: string | null
}
interface LogMessage {
  type: 'stdout' | 'stderr' | 'system'
  line: string
  timestamp: string
  done?: boolean
}
```

## 14. Vite Proxy Configuration

`vite.config.ts` must proxy REST + WebSocket traffic:

```ts
server: {
  proxy: {
    '/api': {
      target: 'http://localhost:8000',
      ws: true
    }
  }
}
```

## 15. Tailwind Styling Guidelines

- Tailwind-only styling
- Light form panel + dark log viewer contrast
- Desktop-first, responsive layout
- Slate/gray palette
- Dropdown uses `shadow-lg`
- Hover/focus transitions for interactive elements